In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from matplotlib.animation import FuncAnimation
import numpy as np
import sys
import argparse
import os
import glob
import utils_report as ru
import utils_behavior as ub
from utils_behavior import *
import cfg
from utils_features import add_hunting

# Check if we're in interactive mode or batch mode
batchmode = False
if "ipykernel_launcher" in sys.argv[0]:
    print("Interactive mode")
else:
    batchmode = True
    print("Batch/CLI mode")
    # Parses the command line arguments below


def get_latest_flat_pkl_file(input_dir="./"):
    pkl_files = glob.glob(input_dir + "/*.pkl")
    pkl_files = [f for f in pkl_files if "flat" in f]
    if not pkl_files:
        raise FileNotFoundError("No .pkl files found in the current directory.")
    latest_pkl_file = max(pkl_files, key=os.path.getctime)
    return latest_pkl_file

default_dir = "/home/raaghav/zfish/onpolicy/custom/forage/results/rmappo-MultiAgentForagingEnv-check/"
if not os.path.exists(default_dir): # Running on cluster
    default_dir = "/n/home04/ramalik/ZFish/onpolicy/custom/forage/results/rmappo-MultiAgentForagingEnv-check"

# outputs_folder = "/srv/marl/raaghav/marl_zfish/rmappo-MultiAgentForagingEnv-1_agent/20250814_153603_1_bao_vd_0.003_fd_10_action_noise_0.1/outputs/"
# outputs_folder = "/home/raaghav/zfish/onpolicy/custom/forage/results/rmappo-MultiAgentForagingEnv-check/20250808_153214_1_bao_efp_0.05_vd_0.002_fd_10/outputs"
# outputs_folder = "/home/raaghav/zfish/onpolicy/custom/forage/results/rmappo-MultiAgentForagingEnv-check/20250808_153214_1_bao_efp_0.05_vd_0.002_fd_10/additional_exps"
outputs_folder = "./results/rmappo-MultiAgentForagingEnv-check/20250916_153414_1_bao_vd_0.006_fdr_10_run_3/outputs"
# outputs_folder = "/n/holylfs06/LABS/krajan_lab/Lab/zfish/sonja_results/results/rmappo-MultiAgentForagingEnv-check/20250918_122018_1_fixed_speed_vd_0.006_fdr_10_run_2_food_0.25_1.0_curric_time_norm_step/outputs"

# ru.get_latest_outputs_folder(default_dir)
# "/srv/marl/satsingh/marl_fish/20241106_182038/outputs/"  # with biting
# "/home/satsingh/srv/marl/satsingh/marl_fish/rmappo-MultiAgentFishEnv-114/outputs/"  # GOOD
# "/home/satsingh/srv/marl/satsingh/marl_fish/20241013_202859/outputs"  # same as rmappo-MultiAgentFishEnv-114
#     "/home/satsingh/srv/marl/satsingh/marl_fish/20241013_202859/outputs/" # BEST
#     "/home/satsingh/srv/marl/satsingh/marl_fish/20241016_202055/outputs/" # OK
#     "/home/satsingh/srv/marl/satsingh/marl_fish/20241016_202056/outputs/" # OK
#     "/home/satsingh/srv/marl/satsingh/marl_fish/20241016_202053/outputs/" # OK

if batchmode:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "outputs_folder",
        default=outputs_folder,
        nargs="?",
    )
    args = parser.parse_args()
    outputs_folder = args.outputs_folder

print(f"Using outputs folder: {outputs_folder}")

Interactive mode
Using outputs folder: ./results/rmappo-MultiAgentForagingEnv-check/20250916_153414_1_bao_vd_0.006_fdr_10_run_3/outputs


In [5]:
# Get all subfolders in additional_exps
additional_exps_dir = os.path.join(outputs_folder, "additional_exps")
if os.path.exists(additional_exps_dir):
    subfolders = [f for f in os.listdir(additional_exps_dir) if os.path.isdir(os.path.join(additional_exps_dir, f))]
    
    # Dictionary to store dataframes from each subfolder
    dff_dict = {}
    
    for subfolder in subfolders:
        subfolder_path = os.path.join(additional_exps_dir, subfolder)
        # try:
        flat_pkl_file = get_latest_flat_pkl_file(subfolder_path)
        dff_dict[subfolder] = pd.read_pickle(flat_pkl_file)
        # Sort the dataframe by the specified columns
        dff_dict[subfolder] = dff_dict[subfolder].sort_values(
            by=["env_id", "episode_index", "agent_id", "time_step"]
        ).reset_index(drop=True)
        print(f"Loaded {subfolder}: {flat_pkl_file}")
        print(f"  Shape: {dff_dict[subfolder].shape}")
        # except FileNotFoundError:
        #     print(f"No .pkl files found in {subfolder}")
    
    print(f"\nLoaded {len(dff_dict)} dataframes from additional_exps subfolders")
else:
    print(f"No additional_exps directory found in {outputs_folder}")
    raise SystemExit


FileNotFoundError: [WinError 3] The system cannot find the path specified: './results/rmappo-MultiAgentForagingEnv-check/20250916_153414_1_bao_vd_0.006_fdr_10_run_3/outputs\\additional_exps\\control\\MAZFish_neural_20250916_153414_bao_vd_0.006_fdr_10_run_3_agg_flattened.pkl'